# 👥 Smart Business Intelligence - Customer Segmentation (RFM & K-Means)

**Projet :** Smart Business Intelligence & Sales Analytics Platform  
**Auteur :** Chaima Debchi  
**Objectif :** Segmenter la base de clients grâce à l'analyse RFM et au clustering non supervisé (K-Means) afin de personnaliser les campagnes marketing et maximiser la fidélisation.

---

## 🧠 Comprendre la Méthodologie RFM
- **Récence (R) :** Nombre de jours écoulés depuis le dernier achat du client. (Plus c'est bas, plus le client est actif).
- **Fréquence (F) :** Nombre total de commandes passées par le client. (Mesure la fidélité).
- **Montant / Valeur Monétaire (M) :** Chiffre d'affaires total généré par le client. (Mesure la valeur financière).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: "%.2f" % x)

df = pd.read_csv("../data/processed/cleaned_superstore.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"])
print(f"Transactions : {len(df):,}, Clients uniques : {df['Customer ID'].nunique():,}")

## 1. Calcul des Métriques RFM par Client

In [ ]:
# Date de référence = lendemain de la dernière commande enregistrée
snapshot_date = df["Order Date"].max() + pd.Timedelta(days=1)
print("Date de référence pour le calcul de la récence :", snapshot_date.date())

rfm = df.groupby("Customer ID").agg({
    "Order Date": lambda x: (snapshot_date - x.max()).days,
    "Order ID": "nunique",
    "Sales": "sum",
    "Profit": "sum",
    "Customer Name": "first"
}).rename(columns={
    "Order Date": "Recency",
    "Order ID": "Frequency",
    "Sales": "Monetary"
}).reset_index()

rfm.head()

## 2. Prétraitement : Transformation Logarithmique et Standardisation

> **Pourquoi la transformation log (`np.log1p`) ?**  
> Les distributions financières ont une forte asymétrie à droite (quelques gros clients dépensent 25 000 $ alors que la médiane est à 2 256 $). La transformation logarithmique resserre les écarts extrêmes pour rendre la distribution plus proche d'une loi normale.
>
> **Pourquoi le `StandardScaler` ?**  
> K-Means utilise la distance euclidienne. Sans standardisation, la variable Monétaire (en milliers de $) écraserait la Fréquence (valeurs entre 1 et 15).

In [ ]:
features = ["Recency", "Frequency", "Monetary"]
rfm_log = np.log1p(rfm[features])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
print("Données normalisées prêtes. Dimensions :", rfm_scaled.shape)

## 3. Détermination du Nombre Optimal de Clusters (K)
Nous évaluons :
1. **L'Inertie (Méthode du Coude / Elbow Method)**
2. **Le Score Silhouette** (mesure la séparation et la cohésion des groupes entre -1 et +1)

In [ ]:
inertia = []
sil_scores = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(k_range, inertia, marker="o", color="#1f77b4")
ax1.set_title("Méthode du Coude (Elbow Method)", fontweight="bold")
ax1.set_xlabel("Nombre de Clusters (K)")
ax1.set_ylabel("Inertie")

ax2.plot(k_range, sil_scores, marker="s", color="#2ca02c")
ax2.set_title("Score Silhouette selon K", fontweight="bold")
ax2.set_xlabel("Nombre de Clusters (K)")
ax2.set_ylabel("Score Silhouette")
plt.tight_layout()
plt.show()

> **Décision :** Nous retenons **K = 4 clusters**. Ce choix offre le meilleur compromis statistique (silhouette maximale de 0.255) et permet une segmentation marketing très actionnable.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

# Synthèse par cluster
cluster_profile = rfm.groupby("Cluster").agg({
    "Recency": ["count", "mean", "median"],
    "Frequency": ["mean", "median"],
    "Monetary": ["mean", "median"],
    "Profit": ["mean", "sum"]
})
cluster_profile

## 4. Interprétation et Nommage Métier des Segments

En analysant les moyennes de chaque cluster, nous attribuons des noms de profils marketing concrets :
- **Champions / High Value :** Dépenses élevées (moyenne 4 773 $), fréquence d'achat forte (8.6 commandes) et profit moyen élevé.
- **Active & Loyal :** Achats très récents (moyenne 20 jours !), fidèles (6.8 commandes) et rentables.
- **At Risk / Slipping :** Clients réguliers dans le passé mais inactifs depuis plusieurs mois (récence moyenne 235 jours).
- **Lost / Dormant :** Inactifs depuis près d'un an (327 jours), faible dépense (432 $), 1 à 2 commandes seulement.

In [ ]:
# Attribution automatique des libellés
means = rfm.groupby("Cluster")[features].mean()
champ_id = means["Monetary"].idxmax()
lost_id = means["Recency"].idxmax()
remaining = [c for c in means.index if c not in [champ_id, lost_id]]
if means.loc[remaining[0], "Recency"] < means.loc[remaining[1], "Recency"]:
    active_id, at_risk_id = remaining[0], remaining[1]
else:
    active_id, at_risk_id = remaining[1], remaining[0]

label_map = {
    champ_id: "Champions / High Value",
    active_id: "Active & Loyal",
    at_risk_id: "At Risk / Slipping",
    lost_id: "Lost / Dormant"
}
rfm["Segment_Name"] = rfm["Cluster"].map(label_map)

# Répartition des clients
print(rfm["Segment_Name"].value_counts())

# Visualisation Scatter Recency vs Monetary
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=rfm,
    x="Recency",
    y="Monetary",
    hue="Segment_Name",
    palette="tab10",
    alpha=0.8,
    s=80
)
plt.yscale("log")
plt.title("Cartographie des Segments Clients (Récence vs Montant dépensé en Log)", fontsize=14, fontweight="bold")
plt.xlabel("Récence (Jours depuis le dernier achat)")
plt.ylabel("Montant Total Dépensé ($ - échelle log)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Stratégies et Recommandations Marketing par Segment

1. **Champions / High Value (258 clients) :**
   - *Action :* Programme VIP exclusif, avant-premières sur les nouveaux produits, gestionnaire de compte dédié.
   - *Ne pas faire :* Éviter les remises agressives inutiles (ils achètent déjà pour la qualité et le service).
2. **Active & Loyal (199 clients) :**
   - *Action :* Ventes croisées (*Cross-selling*) et recommandations personnalisées basées sur leurs catégories préférées.
3. **At Risk / Slipping (253 clients) :**
   - *Action :* Campagnes de réactivation ciblées par email avec offre promotionnelle limitée dans le temps pour relancer l'achat.
4. **Lost / Dormant (83 clients) :**
   - *Action :* Enquête de satisfaction pour comprendre pourquoi ils sont partis, ou offres de reconquête à coût maîtrisé.